##### Importing packages and libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
import math

In [ ]:
sys.path.append(os.path.abspath("../.."))   # Uncomment and run it once (For importing package)
sys.path
from src import utils

##### Settings for Pandas and vizualization

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
sns.set_style('whitegrid')
sns.set_palette('viridis')

##### Loading the dataset

In [ ]:
df = pd.read_csv("../../data/raw/cc_calls.csv")
display(df.head())

Converting column_names to snake_case

In [ ]:
df = utils.convert_columns_to_snake_case(df)

##### Understanding the dataset

In [ ]:
print("Shape:", df.shape)

print("\nInfo:")
df.info()

Missing Values Analysis

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().mean() * 100).round(2)
}).sort_values('Missing %', ascending=False)
missing

In [ ]:
df.nunique()

##### Dropping redundant columns and duplicated rows

Checking if duplicated rows exist

In [ ]:
print("Duplicate rows exists: ", df.duplicated().any())
print("Duplicate rows: ", df.duplicated().sum())

Dropping duplicated rows

In [ ]:
df = df.drop_duplicates()

Dropping rows where co_ref is null

In [ ]:
# As if co_ref is null we cannot join with other dataset
df = df.dropna(subset=['co_ref'])
df.shape

Dropping analyzed call as it contains only 1's

In [ ]:
# No variance in analyzed_call column 
df = df.drop(columns=['analysed_call'])

Dropping call_year column as call_year can be extracted from call_date

In [ ]:
df = df.drop(columns=['call_year'])
df.shape

##### Transforming columns

Converting date columns to common datetime format

In [ ]:
# Converts call_date to common yyyy-mm-dd format
df['call_date'] = pd.to_datetime(df['call_date'], dayfirst=True, errors='coerce')

In [ ]:
calls_per_day = df['call_date'].value_counts().sort_index()

calls_per_day.plot(figsize=(12,5))
plt.title("Number of Calls per Day")
plt.xlabel("Date")
plt.ylabel("Number of Calls")
plt.show()

Identifying categorical and numerical features

In [ ]:
numerical_df = df.select_dtypes(include=['number'])
categorical_df = df.select_dtypes(include=['object', 'category'])

num_cols = numerical_df.columns.tolist()
cat_cols = categorical_df.columns.tolist()

print(f"Numerical: {num_cols}")
print(f"Categorical: {cat_cols}")

Filling categorical nulls with unknown

In [ ]:
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')
    
print("Nulls exist in categorical columns :", df.isnull().values.any())

In [ ]:
for col in cat_cols:
    print(df[col].value_counts())
    print('\n')

Categorical columns vizualization

In [ ]:

cat_imp_cols = [
    'cc_contractor_sentiment',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_call_initiated_by'
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cat_imp_cols):
    top_data = df[col].value_counts().head(10)
    
    small_labels = [
        (str(name)[:13] + "...") if len(str(name)) > 16 else name 
        for name in top_data.index
    ]
    
    sns.barplot(x=small_labels, y=top_data.values, ax=axes[i], palette="viridis")

    axes[i].set_title(f"Top categories in: {col}", fontsize=14, fontweight='bold', pad=15)
    axes[i].tick_params(axis='x', labelrotation=30)
        
    axes[i].set_ylabel("Count")
    axes[i].set_xlabel(None)

plt.tight_layout()
plt.show()

Categorical columns to numerical conversion

In [ ]:
cat_to_num_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score'
]

df[cat_to_num_cols] = df[cat_to_num_cols].apply(pd.to_numeric, errors='coerce')

print("Missing %: ")
for col in cat_to_num_cols:
    print(f'{col}: {(df[col].isnull().mean() * 100).round(2)}')

In [ ]:
df[cat_to_num_cols].corr()

Dropping issues score as High correlation bw overall and issues score

In [ ]:
df = df.drop('cc_contractor_sentiment_issues_score', axis=1)

In [ ]:
cat_to_num_cols = cat_to_num_cols[:3]

for col in cat_to_num_cols:
    plt.figure()
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

Updating start/end/overall sentiment score null values with median

In [ ]:
for col in cat_to_num_cols:
    median = df[col].median()
    df[col] = df[col].fillna(median)
    
df[cat_to_num_cols].isnull().values.any()

Updating cat_cols list after conversion

In [ ]:
numerical_df = df.select_dtypes(include=['number'])
categorical_df = df.select_dtypes(include=['object', 'category'])

num_cols = numerical_df.columns.tolist()
cat_cols = categorical_df.columns.tolist()

print(f"Numerical: {num_cols}")
print(f"Categorical: {cat_cols}")

Cleaning & Standardizing Yes/No Categorical Features

In [ ]:
yes_no_cols = []
valid_categories = {'Yes', 'No', 'Unknown'}

for col in cat_cols:
    unique_vals = df[col].unique()
    
    # Checking if unique vals is superset of valid categories
    if valid_categories.issubset(set(unique_vals)):
        yes_no_cols.append(col)
        
def clean_yes_no(col):
    return col.apply(lambda x: x if x in ['Yes', 'No'] else 'Unknown')

for col in yes_no_cols:
    df[col] = clean_yes_no(df[col])
    
df[yes_no_cols].apply(lambda s: s.value_counts()).T

Updating features where values has multiple categories

In [ ]:
# Different Categorical values
multiple_categories_cols = ['cc_call_initiated_by', 'cc_care_package', 'cc_contractor_sentiment']

cc_care_package_categories = {'Not Discussed', 'Standard', 'Express', 'Premier', 'Unknown', 'Assisted'}
cc_call_initiated_by_categories = {'Customer', 'Agent', 'Unknown', 'Not Relevant'}
cc_contractor_sentiment_categories = {'Satisfied', 'Neutral', 'Not Discussed', 'Unknown', 'Dissatisfied'}

category_map = {
    'cc_care_package': cc_care_package_categories,
    'cc_call_initiated_by': cc_call_initiated_by_categories,
    'cc_contractor_sentiment': cc_contractor_sentiment_categories
}

for col, valid_set in category_map.items():
    df[col] = df[col].apply(lambda x: x if x in valid_set else 'Unknown')

Visualizing multiple categories features after cleaning and standardization

In [ ]:
for col in multiple_categories_cols:
    plt.figure(figsize=(8, 4))
    
    sns.countplot(
        data=df,
        x=col,
        order=df[col].value_counts().index
    )
    
    plt.title(f"Count Plot of {col}")
    plt.tight_layout()
    plt.show()

Final Missing values check

In [ ]:
print("If nulls exists in data: ", df.isnull().sum().any())
print(f"\nFinal shape: {df.shape}")

##### Exporting excel file

In [ ]:
df.to_csv('../../data/interim/cc_calls_cleaned.csv', index=False)